# 🔬 Media Chronicle — YOLO Face Detection Pipeline

This notebook provides an interactive experimentation environment for the entire YOLOv8
face detection pipeline. Each section corresponds to a stage in the pipeline and can be
run independently or sequentially.

**Sections:**
1. Environment Setup & Configuration
2. Dataset Preparation & Exploration
3. Data Augmentation Preview
4. Model Training
5. Training Metrics Visualization
6. Model Evaluation
7. Inference & Detection Visualization
8. Model Export
9. Embedding Analysis & Clustering

---

## §1 — Environment Setup & Configuration

Install dependencies and configure paths. This cell only needs to run once per session.

In [1]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# If running in a uv-managed environment, these should already be available.
# Otherwise, uncomment and run:
#
# !pip install ultralytics>=8.2.0 albumentations>=1.4.0 pyyaml>=6.0 \
#              opencv-python-headless>=4.8.0 Pillow>=10.0 rich>=13.0 \
#              matplotlib>=3.8 seaborn>=0.13 scikit-learn>=1.4

import sys
from pathlib import Path

# ── Add the scripts/ directory to sys.path so we can import our utils ────────
PROJECT_ROOT = Path.cwd()

# Walk upward to find pubspec.yaml if we're running from experiments/
for ancestor in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    if (ancestor / 'pubspec.yaml').exists():
        PROJECT_ROOT = ancestor
        break

SCRIPTS_DIR = PROJECT_ROOT / 'scripts'
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

print(f'Project root: {PROJECT_ROOT}')
print(f'Scripts dir:  {SCRIPTS_DIR}')

Project root: d:\lab\projects\media_chronicle
Scripts dir:  d:\lab\projects\media_chronicle\scripts


In [2]:
# ── Load shared configuration ────────────────────────────────────────────────
from utils.config import YoloConfig
from utils.logging_setup import setup_logger

# Create a configuration instance with experiment-friendly defaults.
# Override any values here for your experiment:
cfg = YoloConfig(
    model_variant='n',          # Start with nano for fast iteration
    epochs=10,                  # Short runs for experimentation
    batch_size=8,               # Lower batch size for limited VRAM
    image_size=640,
    lr=0.01,
    device='cpu',               # Change to '0' for GPU
    verbose=True,
)

logger = setup_logger(verbose=True)
print(cfg.summary())

[06/01/26 13:36:51] DEBUG    Logger initialised — level=10,                                    ]8;id=7568814;file://d:\lab\projects\media_chronicle\scripts\utils\logging_setup.py\logging_setup.py]8;;\:]8;id=7568815;file://d:\lab\projects\media_chronicle\scripts\utils\logging_setup.py#115\115]8;;\
                             file=D:\lab\projects\media_chronicle\runs\logs\yolo_pipeline.log                      

┌─── YOLO Pipeline Configuration ───────────────────────────┐
│  Project root      : D:\lab\projects\media_chronicle
│  Data directory    : D:\lab\projects\media_chronicle\datasets\faces
│  Output directory  : D:\lab\projects\media_chronicle\runs\train
│  Model             : yolov8n (yolov8n.pt)
│  Image size        : 640×640
│  Epochs            : 30
│  Batch size        : 16
│  Learning rate     : 0.01
│  Device            : cpu
│  Workers           : 2
│  Seed              : 42
│  Augmentation      : ON
│  Freeze backbone   : 0 layers
│  Export format     : onnx
│  Verbose           : False
└──────────────────────────────────────────────────────────┘


In [3]:
# ── Common imports ───────────────────────────────────────────────────────────
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from IPython.display import display, HTML

# Configure matplotlib for inline display with dark theme
plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'monospace'

print('Common imports loaded ✓')

Common imports loaded ✓


---
## §2 — Dataset Preparation & Exploration

Scaffold the dataset directory, inspect sample images, and verify label formats.

In [4]:
# ── Scaffold the dataset directory structure ─────────────────────────────────
# This creates datasets/faces/ with train/val subdirectories and a data.yaml.

DATA_DIR = cfg.data_dir
print(f'Dataset directory: {DATA_DIR}')

# Create directory structure
for split in ('train', 'val'):
    for subdir in ('images', 'labels'):
        (DATA_DIR / split / subdir).mkdir(parents=True, exist_ok=True)

# Create data.yaml if it doesn't exist
data_yaml = DATA_DIR / 'data.yaml'
if not data_yaml.exists():
    data_yaml.write_text(
        f'path: {DATA_DIR.resolve()}\n'
        f'train: train/images\n'
        f'val: val/images\n'
        f'\n'
        f'names:\n'
        f'  0: face\n',
        encoding='utf-8'
    )
    print('Created data.yaml ✓')
else:
    print('data.yaml already exists ✓')

# Show directory tree
print(f'\nDirectory structure:')
for p in sorted(DATA_DIR.rglob('*')):
    relative = p.relative_to(DATA_DIR)
    indent = '  ' * len(relative.parts)
    icon = '📁' if p.is_dir() else '📄'
    print(f'{indent}{icon} {p.name}')

Dataset directory: D:\lab\projects\media_chronicle\datasets\faces
Created data.yaml ✓

Directory structure:
  📄 data.yaml
  📁 train
    📁 images
    📁 labels
  📁 val
    📁 images
    📁 labels


In [5]:
# ── Dataset Statistics ────────────────────────────────────────────────────────
# Count images and labels per split.

SUPPORTED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

for split in ('train', 'val'):
    img_dir = DATA_DIR / split / 'images'
    lbl_dir = DATA_DIR / split / 'labels'
    
    images = [f for f in img_dir.iterdir() if f.suffix.lower() in SUPPORTED_EXTENSIONS] if img_dir.exists() else []
    labels = list(lbl_dir.glob('*.txt')) if lbl_dir.exists() else []
    
    # Count total bounding boxes
    total_bboxes = 0
    for lbl in labels:
        lines = lbl.read_text(encoding='utf-8').strip().splitlines()
        total_bboxes += len([l for l in lines if l.strip()])
    
    print(f'\n── {split.upper()} ──')
    print(f'  Images:         {len(images)}')
    print(f'  Label files:    {len(labels)}')
    print(f'  Bounding boxes: {total_bboxes}')


── TRAIN ──
  Images:         0
  Label files:    0
  Bounding boxes: 0

── VAL ──
  Images:         0
  Label files:    0
  Bounding boxes: 0


In [6]:
# ── Visualize Sample Images with Bounding Boxes ──────────────────────────────
# Display a grid of training images with their YOLO label bounding boxes drawn.

def plot_image_with_boxes(img_path: Path, lbl_path: Path, ax, class_names=None):
    """Draw an image with its YOLO bounding boxes overlaid."""
    img = Image.open(img_path)
    img_w, img_h = img.size
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=9, color='#88aaff')
    ax.axis('off')
    
    if lbl_path.exists():
        for line in lbl_path.read_text(encoding='utf-8').strip().splitlines():
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls_id = int(parts[0])
            x_center, y_center, w, h = map(float, parts[1:5])
            
            # Convert YOLO normalised → absolute pixel coordinates
            x1 = (x_center - w/2) * img_w
            y1 = (y_center - h/2) * img_h
            box_w = w * img_w
            box_h = h * img_h
            
            # Draw bounding box
            color = plt.cm.Set1(cls_id % 9)
            rect = patches.Rectangle((x1, y1), box_w, box_h,
                                     linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            
            # Label text
            label = class_names[cls_id] if class_names and cls_id < len(class_names) else f'cls_{cls_id}'
            ax.text(x1, y1 - 4, label, fontsize=8, color=color,
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))


# Collect training images
train_imgs = sorted(
    f for f in (DATA_DIR / 'train' / 'images').iterdir()
    if f.suffix.lower() in SUPPORTED_EXTENSIONS
) if (DATA_DIR / 'train' / 'images').exists() else []

if train_imgs:
    n_show = min(8, len(train_imgs))
    cols = 4
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    axes = axes.flatten() if n_show > 1 else [axes]
    
    for i, img_path in enumerate(train_imgs[:n_show]):
        lbl_path = DATA_DIR / 'train' / 'labels' / (img_path.stem + '.txt')
        plot_image_with_boxes(img_path, lbl_path, axes[i], class_names=['face'])
    
    # Hide empty subplots
    for j in range(n_show, len(axes)):
        axes[j].axis('off')
    
    plt.suptitle('Training Samples with YOLO Bounding Boxes', fontsize=14, color='white')
    plt.tight_layout()
    plt.show()
else:
    print('⚠️ No training images found. Add images to datasets/faces/train/images/ first.')

⚠️ No training images found. Add images to datasets/faces/train/images/ first.


---
## §3 — Data Augmentation Preview

Preview the augmentation pipeline on sample images to verify transforms are sensible
before applying them to the full dataset.

In [ ]:
# ── Build and preview the augmentation pipeline ──────────────────────────────
from utils.data_transforms import build_augmentation_pipeline

pipeline = build_augmentation_pipeline(
    image_size=cfg.image_size,
    flip_horizontal=True,
    brightness_limit=0.2,
    contrast_limit=0.2,
    rotate_limit=15,
    mosaic=False,
)

# Pick the first training image for augmentation preview
if train_imgs:
    sample_img_path = train_imgs[0]
    sample_img = cv2.imread(str(sample_img_path))
    sample_img_rgb = cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB)
    
    # Read corresponding labels
    sample_lbl_path = DATA_DIR / 'train' / 'labels' / (sample_img_path.stem + '.txt')
    bboxes, class_labels = [], []
    if sample_lbl_path.exists():
        for line in sample_lbl_path.read_text(encoding='utf-8').strip().splitlines():
            parts = line.strip().split()
            if len(parts) >= 5:
                class_labels.append(int(parts[0]))
                bboxes.append([float(v) for v in parts[1:5]])
    
    # Generate 8 augmented variants
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    
    # Show original in first position
    axes[0].imshow(sample_img_rgb)
    axes[0].set_title('ORIGINAL', fontsize=10, color='#ffcc00', fontweight='bold')
    axes[0].axis('off')
    
    for i in range(1, 8):
        try:
            augmented = pipeline(
                image=sample_img_rgb.copy(),
                bboxes=bboxes,
                class_labels=class_labels,
            )
            axes[i].imshow(augmented['image'])
            
            # Draw augmented bounding boxes
            h, w = augmented['image'].shape[:2]
            for bbox in augmented['bboxes']:
                x1 = (bbox[0] - bbox[2]/2) * w
                y1 = (bbox[1] - bbox[3]/2) * h
                bw = bbox[2] * w
                bh = bbox[3] * h
                rect = patches.Rectangle((x1, y1), bw, bh,
                                         linewidth=2, edgecolor='lime', facecolor='none')
                axes[i].add_patch(rect)
            
            axes[i].set_title(f'Augmented #{i}', fontsize=9, color='#88ff88')
        except Exception as e:
            axes[i].set_title(f'Failed: {e}', fontsize=8, color='red')
        axes[i].axis('off')
    
    plt.suptitle('Augmentation Pipeline Preview', fontsize=14, color='white')
    plt.tight_layout()
    plt.show()
else:
    print('⚠️ No training images available for augmentation preview.')

---
## §4 — Model Training

Fine-tune a YOLOv8 model on the face dataset. Adjust hyperparameters in the config
cell above before running.

In [ ]:
# ── Train the YOLOv8 model ───────────────────────────────────────────────────
from ultralytics import YOLO

# Load model (downloads COCO pretrained weights if not cached)
model = YOLO(cfg.weights_path)
print(f'Model loaded: {cfg.model_name}')
print(f'Weights: {cfg.weights_path}')

# Verify dataset
data_yaml = cfg.data_dir / 'data.yaml'
assert data_yaml.exists(), f'data.yaml not found at {data_yaml}'
print(f'Dataset: {data_yaml}')

In [ ]:
# ── Execute training ─────────────────────────────────────────────────────────
# ⚠️ This cell will take a while depending on your dataset size and hardware.
# Monitor the output for epoch-by-epoch progress.

OUTPUT_DIR = cfg.output_dir

results = model.train(
    data=str(data_yaml),
    epochs=cfg.epochs,
    batch=cfg.batch_size,
    imgsz=cfg.image_size,
    lr0=cfg.lr,
    momentum=cfg.momentum,
    weight_decay=cfg.weight_decay,
    device=cfg.device,
    workers=cfg.workers,
    seed=cfg.seed,
    augment=cfg.augment,
    freeze=cfg.freeze_backbone if cfg.freeze_backbone > 0 else None,
    project=str(OUTPUT_DIR.parent),
    name=OUTPUT_DIR.name,
    exist_ok=True,
    verbose=True,
    plots=True,
    save=True,
    patience=10,              # Early stopping
)

print('\n✅ Training complete!')

---
## §5 — Training Metrics Visualization

Plot training curves (loss, mAP, precision, recall) from the Ultralytics CSV results.

In [ ]:
# ── Load and plot training results ───────────────────────────────────────────
import csv

# Ultralytics saves results to a CSV file in the training output directory
results_csv = OUTPUT_DIR / 'results.csv'

if results_csv.exists():
    # Parse the CSV
    with open(results_csv, 'r') as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    
    # Clean column names (Ultralytics adds spaces)
    clean_rows = []
    for row in rows:
        clean_rows.append({k.strip(): float(v.strip()) for k, v in row.items() if v.strip()})
    
    if clean_rows:
        epochs = list(range(1, len(clean_rows) + 1))
        
        fig, axes = plt.subplots(2, 2, figsize=(16, 10))
        
        # ── Box Loss ─────────────────────────────────────────────────────
        if 'train/box_loss' in clean_rows[0]:
            train_box = [r.get('train/box_loss', 0) for r in clean_rows]
            val_box = [r.get('val/box_loss', 0) for r in clean_rows]
            axes[0, 0].plot(epochs, train_box, 'o-', color='#ff6b6b', label='Train', markersize=3)
            axes[0, 0].plot(epochs, val_box, 's-', color='#4ecdc4', label='Val', markersize=3)
            axes[0, 0].set_title('Box Loss', color='white', fontweight='bold')
            axes[0, 0].legend()
            axes[0, 0].grid(alpha=0.2)
        
        # ── mAP ─────────────────────────────────────────────────────────
        if 'metrics/mAP50(B)' in clean_rows[0]:
            map50 = [r.get('metrics/mAP50(B)', 0) for r in clean_rows]
            map50_95 = [r.get('metrics/mAP50-95(B)', 0) for r in clean_rows]
            axes[0, 1].plot(epochs, map50, 'o-', color='#45b7d1', label='mAP@50', markersize=3)
            axes[0, 1].plot(epochs, map50_95, 's-', color='#96ceb4', label='mAP@50-95', markersize=3)
            axes[0, 1].set_title('Mean Average Precision', color='white', fontweight='bold')
            axes[0, 1].legend()
            axes[0, 1].grid(alpha=0.2)
        
        # ── Precision ────────────────────────────────────────────────────
        if 'metrics/precision(B)' in clean_rows[0]:
            precision = [r.get('metrics/precision(B)', 0) for r in clean_rows]
            axes[1, 0].plot(epochs, precision, 'o-', color='#ffd93d', label='Precision', markersize=3)
            axes[1, 0].set_title('Precision', color='white', fontweight='bold')
            axes[1, 0].legend()
            axes[1, 0].grid(alpha=0.2)
        
        # ── Recall ───────────────────────────────────────────────────────
        if 'metrics/recall(B)' in clean_rows[0]:
            recall = [r.get('metrics/recall(B)', 0) for r in clean_rows]
            axes[1, 1].plot(epochs, recall, 'o-', color='#ff8a65', label='Recall', markersize=3)
            axes[1, 1].set_title('Recall', color='white', fontweight='bold')
            axes[1, 1].legend()
            axes[1, 1].grid(alpha=0.2)
        
        for ax in axes.flatten():
            ax.set_xlabel('Epoch', color='#888888')
        
        plt.suptitle('Training Metrics Over Epochs', fontsize=16, color='white', fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        # Print final epoch metrics
        last = clean_rows[-1]
        print(f"\n═══ Final Epoch Metrics ═══")
        for key, val in sorted(last.items()):
            print(f"  {key}: {val:.4f}")
    else:
        print('Results CSV is empty.')
else:
    print(f'⚠️ No results.csv found at {results_csv}. Train the model first (§4).')

In [ ]:
# ── Display Ultralytics-generated plots ──────────────────────────────────────
# Ultralytics automatically generates confusion_matrix.png, results.png, etc.

plot_files = [
    'confusion_matrix.png',
    'confusion_matrix_normalized.png',
    'results.png',
    'PR_curve.png',
    'F1_curve.png',
    'P_curve.png',
    'R_curve.png',
]

found_plots = [OUTPUT_DIR / p for p in plot_files if (OUTPUT_DIR / p).exists()]

if found_plots:
    cols = min(3, len(found_plots))
    rows = (len(found_plots) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 6 * rows))
    axes = np.array(axes).flatten() if len(found_plots) > 1 else [axes]
    
    for i, plot_path in enumerate(found_plots):
        img = Image.open(plot_path)
        axes[i].imshow(img)
        axes[i].set_title(plot_path.stem, fontsize=10, color='#88aaff')
        axes[i].axis('off')
    
    for j in range(len(found_plots), len(axes)):
        axes[j].axis('off')
    
    plt.suptitle('Ultralytics Training Plots', fontsize=14, color='white')
    plt.tight_layout()
    plt.show()
else:
    print('⚠️ No training plots found. Train the model first (§4).')

---
## §6 — Model Evaluation

Run formal validation on the held-out validation set and report mAP, precision, and recall.

In [ ]:
# ── Evaluate the trained model ───────────────────────────────────────────────

# Load the best weights from training
best_weights = OUTPUT_DIR / 'weights' / 'best.pt'

if best_weights.exists():
    eval_model = YOLO(str(best_weights))
    print(f'Loaded best weights: {best_weights}')
    
    val_results = eval_model.val(
        data=str(data_yaml),
        imgsz=cfg.image_size,
        batch=cfg.batch_size,
        device=cfg.device,
        verbose=True,
        plots=True,
    )
    
    print('\n═══ Validation Results ═══')
    print(f"  mAP@50:      {val_results.results_dict.get('metrics/mAP50(B)', 0):.4f}")
    print(f"  mAP@50-95:   {val_results.results_dict.get('metrics/mAP50-95(B)', 0):.4f}")
    print(f"  Precision:   {val_results.results_dict.get('metrics/precision(B)', 0):.4f}")
    print(f"  Recall:      {val_results.results_dict.get('metrics/recall(B)', 0):.4f}")
else:
    print(f'⚠️ No trained weights found at {best_weights}. Train the model first (§4).')

---
## §7 — Inference & Detection Visualization

Run the trained model on test images and visualize detected face bounding boxes.

In [ ]:
# ── Run inference on sample images ───────────────────────────────────────────

# Choose source: validation images, or specify a custom path
INFERENCE_SOURCE = cfg.data_dir / 'val' / 'images'

# Load model (best trained, or pretrained fallback)
if best_weights.exists():
    detect_model = YOLO(str(best_weights))
    weight_label = 'trained (best.pt)'
else:
    detect_model = YOLO(cfg.weights_path)
    weight_label = f'pretrained ({cfg.weights_path})'

print(f'Model: {weight_label}')
print(f'Source: {INFERENCE_SOURCE}')

In [ ]:
# ── Detect and visualize ─────────────────────────────────────────────────────

test_images = sorted(
    f for f in INFERENCE_SOURCE.iterdir()
    if f.suffix.lower() in SUPPORTED_EXTENSIONS
) if INFERENCE_SOURCE.exists() else []

if test_images:
    n_show = min(8, len(test_images))
    cols = 4
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(20, 5 * rows))
    axes = axes.flatten() if n_show > 1 else [axes]
    
    for i, img_path in enumerate(test_images[:n_show]):
        # Run prediction
        predictions = detect_model.predict(
            source=str(img_path),
            conf=cfg.confidence_threshold,
            iou=cfg.iou_threshold,
            imgsz=cfg.image_size,
            device=cfg.device,
            verbose=False,
        )
        
        # Load and display original image
        img = Image.open(img_path)
        img_w, img_h = img.size
        axes[i].imshow(img)
        
        # Draw detected bounding boxes
        n_detections = 0
        for result in predictions:
            if result.boxes is not None:
                for j in range(len(result.boxes)):
                    xyxy = result.boxes.xyxy[j].tolist()
                    conf = float(result.boxes.conf[j])
                    cls_id = int(result.boxes.cls[j])
                    cls_name = result.names.get(cls_id, f'cls_{cls_id}')
                    
                    x1, y1, x2, y2 = xyxy
                    rect = patches.Rectangle(
                        (x1, y1), x2 - x1, y2 - y1,
                        linewidth=2, edgecolor='lime', facecolor='none'
                    )
                    axes[i].add_patch(rect)
                    axes[i].text(
                        x1, y1 - 6, f'{cls_name} {conf:.2f}',
                        fontsize=8, color='lime',
                        bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7)
                    )
                    n_detections += 1
        
        axes[i].set_title(f'{img_path.name} ({n_detections} faces)', fontsize=9, color='#88ff88')
        axes[i].axis('off')
    
    for j in range(n_show, len(axes)):
        axes[j].axis('off')
    
    plt.suptitle('YOLOv8 Face Detection Results', fontsize=14, color='white')
    plt.tight_layout()
    plt.show()
else:
    print(f'⚠️ No images found in {INFERENCE_SOURCE}.')

---
## §8 — Model Export

Export the trained model to ONNX or other deployment formats.

In [ ]:
# ── Export the trained model to ONNX ──────────────────────────────────────────

if best_weights.exists():
    export_model = YOLO(str(best_weights))
    
    # Export to ONNX (recommended for Windows desktop deployment)
    exported_path = export_model.export(
        format='onnx',
        imgsz=cfg.image_size,
        device='cpu',
        simplify=True,        # Simplify the ONNX graph
        dynamic=False,        # Fixed input shape for faster inference
        opset=17,
    )
    
    if exported_path:
        export_path = Path(exported_path)
        size_mb = export_path.stat().st_size / (1024 * 1024) if export_path.is_file() else 0
        print(f'\n✅ Model exported to: {export_path}')
        print(f'   Size: {size_mb:.2f} MB')
    else:
        print('⚠️ Export returned no path.')
else:
    print(f'⚠️ No trained weights found at {best_weights}. Train the model first (§4).')

---
## §9 — Embedding Analysis & Clustering

Analyse face embeddings extracted from detections. This mirrors the Dart-side
`YoloEmbeddingsMap` widget and helps verify that identity clusters are well-separated.

In [ ]:
# ── Simulate 2D face embedding clusters ──────────────────────────────────────
# This replicates the Dart-side SingleLayerPerceptron training on synthetic
# face embeddings, allowing you to experiment with cluster parameters.

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Generate synthetic face embeddings (mimicking the Dart-side approach)
np.random.seed(42)

# Define identity clusters with centers similar to the Dart provider
clusters = {
    'John':    {'center': (20, 30), 'n': 15, 'spread': 6},
    'Sarah':   {'center': (70, 80), 'n': 12, 'spread': 6},
    'Unknown': {'center': (50, 50), 'n': 20, 'spread': 25},
}

all_embeddings = []
all_labels = []
all_names = []

for name, params in clusters.items():
    cx, cy = params['center']
    n = params['n']
    spread = params['spread']
    
    embeddings = np.column_stack([
        np.random.normal(cx, spread, n),
        np.random.normal(cy, spread, n),
    ])
    all_embeddings.append(embeddings)
    all_labels.extend([name] * n)
    all_names.append(name)

embeddings = np.vstack(all_embeddings)
labels = np.array(all_labels)

# ── Plot the embedding space ─────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Left: Ground truth clusters
colors = {'John': '#ff6b6b', 'Sarah': '#4ecdc4', 'Unknown': '#888888'}
for name in clusters:
    mask = labels == name
    ax1.scatter(
        embeddings[mask, 0], embeddings[mask, 1],
        c=colors[name], label=name, alpha=0.7, s=60, edgecolors='white', linewidth=0.5
    )

ax1.set_title('Ground Truth Identity Clusters', fontsize=13, color='white', fontweight='bold')
ax1.set_xlabel('Embedding X', color='#888888')
ax1.set_ylabel('Embedding Y', color='#888888')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.15)
ax1.set_xlim(-10, 110)
ax1.set_ylim(-10, 110)

# Right: K-Means clustering (unsupervised discovery)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_ids = kmeans.fit_predict(embeddings)

scatter = ax2.scatter(
    embeddings[:, 0], embeddings[:, 1],
    c=cluster_ids, cmap='Set1', alpha=0.7, s=60, edgecolors='white', linewidth=0.5
)

# Plot cluster centers
centers = kmeans.cluster_centers_
ax2.scatter(centers[:, 0], centers[:, 1], c='yellow', marker='*', s=300,
            edgecolors='white', linewidth=1.5, zorder=5, label='Centroids')

ax2.set_title('K-Means Unsupervised Clustering', fontsize=13, color='white', fontweight='bold')
ax2.set_xlabel('Embedding X', color='#888888')
ax2.set_ylabel('Embedding Y', color='#888888')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.15)
ax2.set_xlim(-10, 110)
ax2.set_ylim(-10, 110)

plt.suptitle('Face Embedding Cluster Analysis', fontsize=16, color='white', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nCluster purity analysis:')
for cluster_id in range(3):
    cluster_mask = cluster_ids == cluster_id
    cluster_labels = labels[cluster_mask]
    unique, counts = np.unique(cluster_labels, return_counts=True)
    dominant = unique[np.argmax(counts)]
    purity = np.max(counts) / len(cluster_labels)
    print(f'  Cluster {cluster_id}: dominant={dominant}, purity={purity:.2%}, size={len(cluster_labels)}')

In [ ]:
# ── Softmax Classifier Training (mirrors Dart SingleLayerPerceptron) ──────────
# Train a simple softmax classifier on the 2D embeddings to replicate
# the Dart-side active learning retraining loop.

class SoftmaxClassifier:
    """Pure-Python multi-class softmax classifier (mirrors Dart implementation)."""
    
    def __init__(self, n_classes: int, n_features: int = 2, seed: int = 42):
        rng = np.random.RandomState(seed)
        self.weights = rng.randn(n_classes, n_features) * 0.1
        self.biases = np.zeros(n_classes)
        self.n_classes = n_classes
    
    def predict(self, x: np.ndarray) -> np.ndarray:
        """Forward pass with softmax activation."""
        x_norm = x / 100.0  # Normalise from [0..100] to [0..1]
        scores = self.weights @ x_norm + self.biases
        # Numerical stability: subtract max
        scores -= np.max(scores)
        exps = np.exp(scores)
        return exps / (np.sum(exps) + 1e-15)
    
    def train_epoch(self, X: np.ndarray, y: np.ndarray, lr: float = 0.05):
        """One epoch of SGD with cross-entropy loss."""
        total_loss = 0.0
        correct = 0
        
        for i in range(len(X)):
            probs = self.predict(X[i])
            target = y[i]
            
            # Cross-entropy loss
            total_loss += -np.log(np.clip(probs[target], 1e-15, 1.0))
            
            # Accuracy
            if np.argmax(probs) == target:
                correct += 1
            
            # SGD update
            x_norm = X[i] / 100.0
            for k in range(self.n_classes):
                dz = probs[k] - (1.0 if k == target else 0.0)
                self.weights[k] -= lr * dz * x_norm
                self.biases[k] -= lr * dz
        
        return {
            'loss': total_loss / len(X),
            'accuracy': correct / len(X),
        }


# Filter to only identified faces (John and Sarah)
identified_mask = labels != 'Unknown'
train_X = embeddings[identified_mask]
train_names = labels[identified_mask]
unique_names = sorted(set(train_names))
name_to_idx = {n: i for i, n in enumerate(unique_names)}
train_y = np.array([name_to_idx[n] for n in train_names])

print(f'Training on {len(train_X)} samples, {len(unique_names)} classes: {unique_names}')

# Train for 15 epochs (matching the Dart-side retrainer)
clf = SoftmaxClassifier(n_classes=len(unique_names))
history = {'loss': [], 'accuracy': []}

for epoch in range(15):
    metrics = clf.train_epoch(train_X, train_y, lr=0.05)
    history['loss'].append(metrics['loss'])
    history['accuracy'].append(metrics['accuracy'])
    print(f"  Epoch {epoch+1:2d}/15 — loss: {metrics['loss']:.4f}, accuracy: {metrics['accuracy']*100:.1f}%")

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(1, 16), history['loss'], 'o-', color='#ff6b6b', markersize=5)
ax1.set_title('Cross-Entropy Loss', fontsize=12, color='white', fontweight='bold')
ax1.set_xlabel('Epoch', color='#888888')
ax1.grid(alpha=0.2)

ax2.plot(range(1, 16), [a * 100 for a in history['accuracy']], 'o-', color='#4ecdc4', markersize=5)
ax2.set_title('Training Accuracy (%)', fontsize=12, color='white', fontweight='bold')
ax2.set_xlabel('Epoch', color='#888888')
ax2.set_ylim(0, 105)
ax2.grid(alpha=0.2)

plt.suptitle('Softmax Classifier Training (mirrors Dart SingleLayerPerceptron)',
             fontsize=13, color='white')
plt.tight_layout()
plt.show()

In [ ]:
# ── Decision Boundary Visualization ──────────────────────────────────────────
# Plot the softmax classifier's decision boundary over the 2D embedding space.

# Create a mesh grid over the embedding space
x_min, x_max = -10, 110
y_min, y_max = -10, 110
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                      np.linspace(y_min, y_max, 300))

# Predict class probabilities for each grid point
grid_points = np.column_stack([xx.ravel(), yy.ravel()])
predictions = np.array([np.argmax(clf.predict(p)) for p in grid_points])
predictions = predictions.reshape(xx.shape)

# Plot
fig, ax = plt.subplots(figsize=(12, 10))

# Decision regions
ax.contourf(xx, yy, predictions, alpha=0.3, cmap='Set1', levels=np.arange(-0.5, len(unique_names) + 0.5))
ax.contour(xx, yy, predictions, colors='white', linewidths=1, alpha=0.5)

# Scatter plot of actual embeddings
scatter_colors = {'John': '#ff6b6b', 'Sarah': '#4ecdc4', 'Unknown': '#888888'}
for name in clusters:
    mask = labels == name
    marker = 'o' if name != 'Unknown' else 'x'
    ax.scatter(
        embeddings[mask, 0], embeddings[mask, 1],
        c=scatter_colors[name], label=name, alpha=0.8, s=80,
        edgecolors='white', linewidth=0.8, marker=marker,
    )

ax.set_title('Softmax Decision Boundary with Face Embeddings',
             fontsize=14, color='white', fontweight='bold')
ax.set_xlabel('Embedding X', fontsize=11, color='#888888')
ax.set_ylabel('Embedding Y', fontsize=11, color='#888888')
ax.legend(fontsize=11, loc='upper left')
ax.grid(alpha=0.15)
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)

plt.tight_layout()
plt.show()

# ── Classify unknown faces ───────────────────────────────────────────────────
unknown_mask = labels == 'Unknown'
unknown_embeddings = embeddings[unknown_mask]

print(f'\nClassifying {len(unknown_embeddings)} unknown faces:')
for i, emb in enumerate(unknown_embeddings):
    probs = clf.predict(emb)
    predicted_idx = np.argmax(probs)
    predicted_name = unique_names[predicted_idx]
    confidence = probs[predicted_idx] * 100
    
    status = '✅' if confidence >= 82 else '❓'
    print(f'  {status} Unknown #{i+1:2d} → {predicted_name} ({confidence:.1f}%)')

---

## 🧪 Experiment Notes

Use this section to document your experimental findings, parameter sweeps,
and observations.

| Experiment | Model | Epochs | LR    | mAP@50 | Notes |
|------------|-------|--------|-------|--------|-------|
| Baseline   | yolov8n | 30   | 0.01  | —      | First run |
| —          | —     | —      | —     | —      | — |